In [1]:
import pandas as pd

In [2]:
events_df = pd.read_csv('../data/raw/events.csv')

In [3]:
funnel_counts = events_df['event'].value_counts()

funnel_order = [
    'signup',
    'activation',
    'feature_use',
    'purchase'
]

funnel_counts = funnel_counts[funnel_order]

In [5]:
funnel_df = pd.DataFrame({

    'Stage': funnel_order,

    'Users': funnel_counts.values
})

funnel_df

,Stage,Users
0,signup,5000
1,activation,3303
2,feature_use,1662
3,purchase,485


In [6]:
funnel_df.to_csv(
    '../data/processed/funnel_summary.csv',
    index=False
)

In [7]:
events_df['timestamp'] = pd.to_datetime(
    events_df['timestamp']
)

events_df['event_date'] = (
    events_df['timestamp'].dt.date
)

In [8]:
signup_dates = events_df[
    events_df['event'] == 'signup'
][['user_id', 'event_date']]

signup_dates.columns = [
    'user_id',
    'signup_date'
]

In [11]:
events_df = events_df.merge(
    signup_dates,
    on='user_id'
)


In [12]:
events_df['days_since_signup'] = (

    pd.to_datetime(events_df['event_date']) -

    pd.to_datetime(events_df['signup_date'])

).dt.days

In [13]:
retention = events_df.groupby(
    'days_since_signup'
)['user_id'].nunique()

retention_df = pd.DataFrame({

    'Day': retention.index,

    'Active Users': retention.values
})

In [14]:
retention_df.to_csv(
    '../data/processed/retention_summary.csv',
    index=False
)


In [15]:
purchase_users = events_df[
    events_df['event'] == 'purchase'
]

purchase_counts = purchase_users.groupby(
    'variant'
)['user_id'].nunique()

total_users = events_df.groupby(
    'variant'
)['user_id'].nunique()

In [16]:
ab_df = pd.DataFrame({

    'Variant': ['A', 'B'],

    'Total Users': [
        total_users['A'],
        total_users['B']
    ],

    'Purchases': [
        purchase_counts['A'],
        purchase_counts['B']
    ]
})

In [17]:
ab_df['Conversion Rate'] = (

    ab_df['Purchases'] /

    ab_df['Total Users']

) * 100

In [18]:
ab_df.to_csv(
    '../data/processed/ab_summary.csv',
    index=False
)